# PM / CNH FD + Dashboard + Part Lookup — 5 Sheet Combined

This notebook combines the existing **PM_CNH_FD_Combined**, **Dashboard Preliminary 1.2.2**, and **Part Lookup Generator 3Aug26** workflows.

## Run settings
**Everything you normally need to change before each run is now in the first code cell.**

This includes:
- PM/CNH mode
- input file names, sheets, and skiprows
- DN Price comparison file settings
- RC multipliers
- output filename
- OCLT
- branch LT values
- Part Lookup price-source behavior

After editing the first code cell, you can run the notebook from top to bottom.

## Final Excel workbook

The output contains exactly these 5 sheets:

1. **Dashboard** — Dashboard Preliminary-style dashboard
2. **Part Lookup** — Part Lookup + demand/calls trend
3. **National Dashboard** — National Dashboard
4. **FD Processed** — PM/CNH FD processing result
5. **pmovdcE** — pmovdcE movement data **plus** the Dashboard Preliminary Data table

### Sheet 5
Because the original pmovdcE does not contain a National branch, this notebook adds a **National** row for every P/N by aggregating the real branch movement history.

The Data table is placed to the right of the pmovdcE table on the same sheet.

- **OPM / SL / PF:** Excel formulas generated from the OPM / PF parameters in the first code cell
- **LT / OCLT:** live Excel formulas
- **ExDlt / OC / Min / ROP / Max / Condition:** live Excel formulas based on LT/OCLT
- Yellow LT/OCLT parameter cells are available on Sheet 5 so they can be changed directly in Excel.

> The source Dashboard Preliminary notebook does not define a formula that derives LT or OCLT; it treats them as inputs. Therefore this merged version keeps the LT/OCLT inputs editable while making their use in the Data table live.


The final workbook builder also normalizes Brc/Agc/P/N join keys so numeric Excel values such as `19`, `19.0`, and `"19"` do not cause Pandas merge dtype errors.


In [ ]:
# ============================================================
# COMBINED PM / CNH FD PROCESS
# ============================================================
# Set this to False for SINGLE AGENCY (PM)
# Set this to True  for MULTIPLE AGENCY (CNH / National)
# ============================================================

import os
import re

MULTIPLE_AGENCY = False

# ============================================================
# FILE CONFIGURATION
# ============================================================
# SINGLE AGENCY files (used when MULTIPLE_AGENCY = False)
SINGLE_PARTS_MOVEMENT_FILE = "pmovdcE agc 19 1Sept26.xlsx"
SINGLE_PARTS_MOVEMENT_SHEET = "Non Steron"
SINGLE_PARTS_MOVEMENT_SKIPROWS = 4
SINGLE_FORECAST_FILE = "FD Non Steron Agc 19 1Sept26.xlsx"
SINGLE_FORECAST_SHEET = "National"

# MULTIPLE AGENCY files (used when MULTIPLE_AGENCY = True)
MULTI_PARTS_MOVEMENT_FILE = "pmovdcE CNH 1Sept26.xlsx"
MULTI_PARTS_MOVEMENT_SHEET = "Non Steron"
MULTI_PARTS_MOVEMENT_SKIPROWS = 4
MULTI_FORECAST_FILE = "FD Non Steron CNH 1Sept26.xlsx"
MULTI_FORECAST_SHEET = "National"

# ============================================================
# PRICE UPDATE SOURCE — USED ONLY FOR MULTIPLE AGENCY
# ============================================================
PRICE_COMPARISON_FILE = "Comparison DN Price CNH 18Aug26.xlsx"
PRICE_COMPARISON_SHEET = "Price List"
PRICE_COMPARISON_HEADER_ROW = 2
PRICE_COMPARISON_PN_COL = "V"
PRICE_COMPARISON_PRICE_COL = "X"

# ============================================================
# RC MULTIPLIERS — EDIT THESE MANUALLY BEFORE EACH RUN
# ============================================================
# These are intentionally NOT changed automatically by the mode.
# Enter the multiplier you want to use for the current run.
RC_MULTIPLIER = {
    "A": 5,
    "B": 4,
    "C": 2.5,
    "D": 0
}

# ============================================================
# OPM / SL / PF PARAMETERS — EDIT HERE BEFORE EACH RUN
# ============================================================
# These values are applied when the notebook is rerun; changing them here
# does not retroactively change an already-created Excel workbook.
opm_quadrants = {
    "A": {"calls_max": 6, "brackets": [(0, 11, 93), (11, 44, 90), (44, 119, 85), (119, 483, 83), (483, 999999, 80)]},
    "B": {"calls_max": 11, "brackets": [(0, 11, 96), (11, 44, 95), (44, 119, 91), (119, 483, 87), (483, 999999, 81)]},
    "C": {"calls_max": 26, "brackets": [(0, 11, 98), (11, 44, 96), (44, 119, 94), (119, 483, 92), (483, 999999, 87)]},
    "D": {"calls_max": 999999999, "brackets": [(0, 11, 99), (11, 44, 97), (44, 119, 95), (119, 483, 93), (483, 999999, 89)]},
}
pf_lookup_values = [80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99]
pf_lookup_results = [0.85,0.88,0.92,0.96,1,1.04,1.09,1.13,1.18,1.23,1.29,1.35,1.41,1.48,1.56,1.65,1.76,1.89,2.06,2.33]

# ============================================================
# OUTPUT / INVENTORY PARAMETERS — EDIT HERE BEFORE EACH RUN
# ============================================================
OUTPUT_FILE = None  # If None, derive from the selected parts-movement source file.

def derive_output_filename(parts_movement_file):
    base = os.path.splitext(os.path.basename(parts_movement_file))[0]
    suffix = re.sub(r"^pmovdce[\s_-]*", "", base, flags=re.IGNORECASE)
    suffix = re.sub(r"^pmovdc[\s_-]*", "", suffix, flags=re.IGNORECASE)
    suffix = re.sub(r"[\s_-]+", "_", suffix).strip("_")
    return f"All_Dashboard_{suffix}.xlsx"


# OCLT used for every row unless changed below.
OCLT_VALUE = 14

# Lead Time by branch. Add/update the branches you need for this run.
# Example: 21: 30 means Branch 21 has LT = 30 days.
BRANCH_LT = {
    # 21: 30,
    # 22: 30,
    # 27: 30,
    # "National": 30,
}

# Part Lookup source behavior. Usually keep this automatic.
USE_PRICE_COMPARISON_FOR_LOOKUP = MULTIPLE_AGENCY

# ============================================================
# SELECT FILES BASED ON MODE
# ============================================================
if MULTIPLE_AGENCY:
    PARTS_MOVEMENT_FILE = MULTI_PARTS_MOVEMENT_FILE
    PARTS_MOVEMENT_SHEET = MULTI_PARTS_MOVEMENT_SHEET
    PARTS_MOVEMENT_SKIPROWS = MULTI_PARTS_MOVEMENT_SKIPROWS
    FORECAST_FILE = MULTI_FORECAST_FILE
    FORECAST_SHEET = MULTI_FORECAST_SHEET
    print("MODE: MULTIPLE AGENCY")
else:
    PARTS_MOVEMENT_FILE = SINGLE_PARTS_MOVEMENT_FILE
    PARTS_MOVEMENT_SHEET = SINGLE_PARTS_MOVEMENT_SHEET
    PARTS_MOVEMENT_SKIPROWS = SINGLE_PARTS_MOVEMENT_SKIPROWS
    FORECAST_FILE = SINGLE_FORECAST_FILE
    FORECAST_SHEET = SINGLE_FORECAST_SHEET
    print("MODE: SINGLE AGENCY")

if OUTPUT_FILE is None:
    OUTPUT_FILE = derive_output_filename(PARTS_MOVEMENT_FILE)

print(f"Parts Movement : {PARTS_MOVEMENT_FILE}")
print(f"Forecast       : {FORECAST_FILE}")

# ============================================================
# LOAD FILES
# ============================================================
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils import column_index_from_string

PartsMovement_df = pd.read_excel(
    PARTS_MOVEMENT_FILE,
    skiprows=PARTS_MOVEMENT_SKIPROWS,
    sheet_name=PARTS_MOVEMENT_SHEET
)

Forecast_df = pd.read_excel(
    FORECAST_FILE,
    sheet_name=FORECAST_SHEET
)

# --- Make P/N case-insensitive ---
PartsMovement_df["P/N"] = PartsMovement_df["P/N"].astype(str).str.upper()

# ============================================================
# UPDATE DN PRICE — MULTIPLE AGENCY ONLY
# ============================================================
def load_price_updates(path, sheet, header_row, pn_col_letter, price_col_letter):
    wb_price = load_workbook(path, read_only=True, data_only=True)
    ws_price = wb_price[sheet]
    pn_idx = column_index_from_string(pn_col_letter)
    price_idx = column_index_from_string(price_col_letter)

    updates = {}
    for row in ws_price.iter_rows(min_row=header_row + 1, values_only=True):
        pn = row[pn_idx - 1]
        price = row[price_idx - 1]
        if pn is None or price is None:
            continue
        updates[str(pn).strip().upper()] = price

    wb_price.close()
    return updates

if MULTIPLE_AGENCY:
    price_updates = load_price_updates(
        PRICE_COMPARISON_FILE,
        PRICE_COMPARISON_SHEET,
        PRICE_COMPARISON_HEADER_ROW,
        PRICE_COMPARISON_PN_COL,
        PRICE_COMPARISON_PRICE_COL
    )

    has_update = PartsMovement_df["P/N"].isin(price_updates)
    PartsMovement_df.loc[has_update, "DN Price"] = (
        PartsMovement_df.loc[has_update, "P/N"].map(price_updates)
    )

    print(
        f"DN Price updated for {has_update.sum():,} of {len(PartsMovement_df):,} rows "
        f"using {PRICE_COMPARISON_FILE}"
    )

# ============================================================
# IDENTIFY COLUMNS
# ============================================================
c_columns = sorted(
    [
        col for col in PartsMovement_df
        if isinstance(col, str) and col.startswith("C-")
    ],
    key=lambda x: int(x.split("-")[1]),
    reverse=True
)

oh_col = "OH"
oo_col = "OO"
dn_price_col = "DN Price"

# ============================================================
# GROUPING — THE MAIN DIFFERENCE BETWEEN THE TWO MODES
# ============================================================
sum_columns = c_columns + [oh_col, oo_col]
aggregation_dict = {col: "sum" for col in sum_columns}
aggregation_dict[dn_price_col] = "first"

if MULTIPLE_AGENCY:
    # Multiple agency: combine all agencies into one P/N-level result.
    df_sum = PartsMovement_df.groupby(
        "P/N", as_index=False
    ).agg(aggregation_dict)
else:
    # Single agency: group by agency + P/N, then remove Agc.
    df_sum = PartsMovement_df.groupby(
        ["Agc", "P/N"], as_index=False
    ).agg(aggregation_dict)

# --- Create Total Calls ---
df_sum["Total Calls"] = df_sum[c_columns].sum(axis=1)

# --- Remove individual C- columns ---
df_sum = df_sum.drop(columns=c_columns)

# --- Multiple agency identifiers ---
if MULTIPLE_AGENCY:
    df_sum.insert(0, "Agc", "All")
    df_sum.insert(0, "Brc", "National")
else:
    # Preserve PM behavior: remove individual agency after grouping.
    df_sum = df_sum.drop(columns=["Agc"])

# --- Reorder columns ---
base_columns = ["P/N", "Total Calls", "DN Price", "OH", "OO"]
if MULTIPLE_AGENCY:
    df_sum = df_sum[["Brc", "Agc"] + base_columns]
else:
    df_sum = df_sum[base_columns]

# --- Sort ---
df_sum = df_sum.sort_values(
    by="Total Calls", ascending=False
).reset_index(drop=True)

cols_to_round = ["Total Calls", "OH", "OO"]
df_sum[cols_to_round] = (
    df_sum[cols_to_round]
    .round(0)
    .astype("Int64")
)


In [2]:
#Perhitungan RC (Rank Call)
# Sort by Total Calls descending
df_sum = (
    df_sum
    .sort_values(
        by=["Total Calls", "P/N"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)


# Add cumulative sum
df_sum["Accum."] = df_sum["Total Calls"].cumsum()

# Percentage cumulative
last_accum = df_sum["Accum."].iloc[-1]
df_sum["%Accum."] = (df_sum["Accum."] / last_accum) * 100
df_sum["%Accum."] = df_sum["%Accum."].round(2)

#klasifikasi RC ABCD
def assign_rc(pct, total_calls):
    if total_calls > 0 and pct >= 100:
        return "C"

    if 0 <= pct <= 50:
        return "A"
    elif 50 < pct <= 80:
        return "B"
    elif 80 < pct < 100:
        return "C"
    else:
        return "D"


df_sum["RC"] = df_sum.apply(
    lambda row: assign_rc(row["%Accum."], row["Total Calls"]),
    axis=1
)


In [3]:
# ============================================================
# MERGE FORECAST
# ============================================================

if MULTIPLE_AGENCY:
    # Multiple agency: use NATIONAL + ALL AGC forecast only.
    Forecast_df["p/n"] = Forecast_df["p/n"].astype(str).str.upper()
    Forecast_df["brc"] = Forecast_df["brc"].astype(str).str.upper()
    Forecast_df["agc"] = Forecast_df["agc"].astype(str).str.upper()

    df_sum["P/N"] = df_sum["P/N"].astype(str).str.upper()

    forecast_filtered = Forecast_df[
        (Forecast_df["brc"] == "NATIONAL") &
        (Forecast_df["agc"] == "ALL AGC")
    ]

    forecast_subset = forecast_filtered[
        ["p/n", "FD_final"]
    ].rename(columns={"p/n": "P/N"})

else:
    # Single agency: use the forecast file directly by P/N.
    if "p/n" in Forecast_df.columns:
        Forecast_df = Forecast_df.rename(columns={"p/n": "P/N"})

    Forecast_df["P/N"] = Forecast_df["P/N"].astype(str).str.upper()
    df_sum["P/N"] = df_sum["P/N"].astype(str).str.upper()

    forecast_subset = Forecast_df[["P/N", "FD_final"]]

# --- Merge into df_sum ---
df_sum = df_sum.merge(
    forecast_subset,
    on="P/N",
    how="left"
)

# --- Move FD_final to the main calculation area ---
fd_val = df_sum.pop("FD_final")
df_sum["FD_final"] = fd_val


In [4]:
#Urutan Kolom df_sum
# --- Define your desired final column order ---
final_columns = [
    "P/N",
    "Total Calls",
    "RC",
    "DN Price",
    "FD_final",
    "OH",
    "OO",
    "Accum.",
    "%Accum."
]

# --- Reorder df_sum (only keep those that exist) ---
df_sum = df_sum[final_columns]

In [5]:
# ============================================================
# PERHITUNGAN MAX
# ============================================================
# RC_MULTIPLIER is defined at the top of the notebook.
# EDIT IT MANUALLY BEFORE EACH RUN.
# It is intentionally NOT changed automatically between modes.

df_sum["RC_value"] = df_sum["RC"].map(RC_MULTIPLIER)
df_sum["Max"] = df_sum["FD_final"] * df_sum["RC_value"]
df_sum["Max"] = df_sum["Max"].round(0)
df_sum = df_sum.drop(columns=["RC_value"])

# ============================================================
# COLUMN ORDER
# ============================================================
final_columns = [
    "P/N",
    "Total Calls",
    "RC",
    "DN Price",
    "FD_final",
    "Max",
    "OH",
    "OO",
    "Accum.",
    "%Accum."
]

df_sum = df_sum[final_columns]


In [6]:
#INCOMING
# List of new incoming columns
incoming_cols = [f"Incoming M-{i}" for i in range(1, 8)]

# Insert after OO
oo_index = df_sum.columns.get_loc("OO")

for i, col_name in enumerate(incoming_cols):
    if col_name not in df_sum.columns:        # <-- Prevent duplicate error
        df_sum.insert(oo_index + 1 + i, col_name, "")


In [7]:
#Estimated OH & Estimated OO
import numpy as np

# ----- CREATE COLUMN NAMES -----
est_oh_cols = [f"Estimated OH M-{i}" for i in range(1, 10)]   # M-1 to M-9
est_oo_cols = [f"Estimated OO M-{i}" for i in range(1, 9)]    # M-1 to M-8

# Insert Estimated OH columns after Incoming M-7
insert_pos = df_sum.columns.get_loc("Incoming M-7") + 1
for i, col in enumerate(est_oh_cols):
    if col not in df_sum.columns:
        df_sum.insert(insert_pos + i, col, np.nan)

# Insert Estimated OO columns after Estimated OH M-9
insert_pos = df_sum.columns.get_loc("Estimated OH M-9") + 1
for i, col in enumerate(est_oo_cols):
    if col not in df_sum.columns:
        df_sum.insert(insert_pos + i, col, np.nan)

# ----- CALCULATE VALUES ROW-BY-ROW -----
for idx, row in df_sum.iterrows():

    # Get fixed values
    OH0 = row["OH"]
    OO0 = row["OO"]
    FD = row["FD_final"]
    Max = row["Max"]

    # Store results
    est_oh = {}
    est_oo = {}

    # ----- Estimated OH M-1 -----
    incoming_1 = float(row["Incoming M-1"]) if row["Incoming M-1"] not in ["", None, np.nan] else 0
    est_oh[1] = (OH0 + OO0 + incoming_1) - FD

    # ----- Estimated OO M-1 -----
    est_oo[1] = 0 if est_oh[1] > Max else (Max - est_oh[1])

    # ----- M-2 to M-9 for Estimated OH, M-2 to M-8 for Estimated OO -----
    for i in range(2, 10):

        incoming_i = (
            float(row[f"Incoming M-{i}"])
            if (f"Incoming M-{i}" in df_sum.columns and row[f"Incoming M-{i}"] not in ["", None, np.nan])
            else 0
        )

        # Estimated OH M-i
        prev_oh = est_oh[i-1]
        prev_oo = est_oo[i-1] if i-1 in est_oo else 0
        est_oh[i] = (prev_oh + incoming_i + prev_oo) - FD

        # Estimated OO only until M-8
        if i <= 8:
            est_oo[i] = 0 if est_oh[i] > Max else (Max - est_oh[i])

    # ----- Assign to dataframe -----
    for i in range(1, 10):
        df_sum.loc[idx, f"Estimated OH M-{i}"] = est_oh[i]

    for i in range(1, 9):
        df_sum.loc[idx, f"Estimated OO M-{i}"] = est_oo[i]


In [8]:
#Schedule Order
import numpy as np

# ----- CREATE COLUMN NAMES -----
schedule_cols = [f"Schedule Order M-{i}" for i in range(1, 7)]

# Insert Schedule Order columns after Estimated OO M-8 (the last OO column)
insert_pos = df_sum.columns.get_loc("Estimated OO M-8") + 1
for i, col in enumerate(schedule_cols):
    if col not in df_sum.columns:
        df_sum.insert(insert_pos + i, col, np.nan)

# ----- CALCULATE SCHEDULE ORDER -----
for idx, row in df_sum.iterrows():

    Max = row["Max"]

    for i in range(1, 7):

        # Lookahead month: OH M-(i+3)
        oh_col = f"Estimated OH M-{i+3}"

        # If OH value exists, fetch it, else use 0
        oh_value = row[oh_col] if oh_col in df_sum.columns else 0

        # Schedule Order formula
        if oh_value > Max:
            sched = 0
        else:
            sched = Max - oh_value

        df_sum.loc[idx, f"Schedule Order M-{i}"] = sched

In [9]:
#Amount Schedule Order
import numpy as np

# ----- CREATE COLUMN NAMES -----
amount_cols = [f"Amount Schedule Order M-{i}" for i in range(1, 7)]

# Find the position after "Schedule Order M-6"
insert_pos = df_sum.columns.get_loc("Schedule Order M-6") + 1

# Insert empty columns first (if not already present)
for i, col in enumerate(amount_cols):
    if col not in df_sum.columns:
        df_sum.insert(insert_pos + i, col, np.nan)

# ----- CALCULATE AMOUNTS -----
for idx, row in df_sum.iterrows():

    dn_price = row["DN Price"]

    for i in range(1, 7):

        sched_col = f"Schedule Order M-{i}"
        amount_col = f"Amount Schedule Order M-{i}"

        schedule_value = row[sched_col] if sched_col in df_sum.columns else 0

        df_sum.loc[idx, amount_col] = schedule_value * dn_price
        # Round ONLY the Amount Schedule Order columns
for col in amount_cols:
    df_sum[col] = df_sum[col].round(2)

In [10]:
# ============================================================
# DESCRIPTION
# ============================================================
# Keep one description per P/N. If several descriptions exist,
# use the longest non-empty description, matching the CNH logic.

desc_df = (
    PartsMovement_df[["P/N", "Desc"]]
    .dropna(subset=["Desc"])
    .assign(len_desc=lambda x: x["Desc"].astype(str).str.len())
    .sort_values("len_desc", ascending=False)
    .drop_duplicates("P/N")[["P/N", "Desc"]]
)

df_sum = df_sum.merge(desc_df, on="P/N", how="left")

insert_pos = df_sum.columns.get_loc("Amount Schedule Order M-6") + 1
desc_series = df_sum.pop("Desc")
df_sum.insert(insert_pos, "Desc", desc_series)


In [11]:
#MM06
c06_cols = [f"C-{i}" for i in range(1, 7)]  # C-1 to C-6
c06_cols = [col for col in c06_cols if col in PartsMovement_df.columns]  # verify exist
# Group by P/N and sum first
temp = PartsMovement_df.groupby("P/N", as_index=False)[c06_cols].sum()

# Count how many months have calls > 0
temp["MM06"] = temp[c06_cols].gt(0).sum(axis=1)

# Keep only P/N + MM06
mm06_df = temp[["P/N", "MM06"]]
df_sum = df_sum.merge(mm06_df, on="P/N", how="left")
insert_pos = df_sum.columns.get_loc("Total Calls") + 1
mm06_series = df_sum.pop("MM06")
df_sum.insert(insert_pos, "MM06", mm06_series)

In [12]:
#MM12
c12_cols = [f"C-{i}" for i in range(1, 13)]  # C-1 to C-12
c12_cols = [col for col in c12_cols if col in PartsMovement_df.columns]  # ensure exist
temp12 = PartsMovement_df.groupby("P/N", as_index=False)[c12_cols].sum()

# Count how many months have calls > 0 (non-zero values)
temp12["MM12"] = temp12[c12_cols].gt(0).sum(axis=1)

# Keep only P/N + MM12
mm12_df = temp12[["P/N", "MM12"]]
df_sum = df_sum.merge(mm12_df, on="P/N", how="left")
insert_pos = df_sum.columns.get_loc("MM06") + 1
mm12_series = df_sum.pop("MM12")
df_sum.insert(insert_pos, "MM12", mm12_series)

In [13]:
#Amount Estimated OO
est_oo_cols = [col for col in df_sum.columns if col.startswith("Estimated OO M-")]
est_oo_cols = sorted(
    est_oo_cols, 
    key=lambda x: int(x.split("M-")[1])  # ensure correct order M-1, M-2, ...
)
# Create monetary columns: DN Price × Estimated OO M-i
for col in est_oo_cols:
    month_num = col.split("M-")[1]  # extracts "1", "2", ... "8"
    new_col = f"Amount Estimated OO M-{month_num}"
    df_sum[new_col] = df_sum["DN Price"] * df_sum[col]
# Find insertion point (after MM12)
insert_pos = df_sum.columns.get_loc("MM12") + 1

# Collect new amount columns in the same order
est_oo_amount_cols = [f"Amount Estimated OO M-{i}" for i in range(1, 9)]

# Move them into correct position
for i, col in enumerate(est_oo_amount_cols):
    series = df_sum.pop(col)
    df_sum.insert(insert_pos + i, col, series)


In [14]:
# --- RENAME COLUMNS FIRST ---

# Create a rename dictionary
rename_dict = {}

for col in df_sum.columns:
    new_col = col
    new_col = new_col.replace("Estimated", "Est.")
    new_col = new_col.replace("Amount", "Amt.")
    new_col = new_col.replace("Schedule","Sched.")
    rename_dict[col] = new_col

# Apply renaming
df_sum = df_sum.rename(columns=rename_dict)


# --- NOW REBUILD THE COLUMN ORDER USING UPDATED NAMES ---

# STARTING COLUMNS
desired_order_start = [
    "P/N",
    "Total Calls",
    "RC",
    "DN Price",
    "FD_final",
    "Max",
    "OH",
    "OO"
]

# Dynamic column groups (renamed ones)
incoming_cols = [col for col in df_sum.columns if col.startswith("Incoming M-")]
est_oh_cols   = [col for col in df_sum.columns if col.startswith("Est. OH M-")]
est_oo_cols   = [col for col in df_sum.columns if col.startswith("Est. OO M-")]
schedule_cols = [col for col in df_sum.columns if col.startswith("Sched. Order M-")]
amt_cols      = [col for col in df_sum.columns if col.startswith("Amt. Sched. Order M-") or col.startswith("Amt. Est. OO")]

ending_cols = [
    "Desc",
    "MM06",
    "MM12",
    "Accum.",
    "%Accum."
]

# Final order (only include columns that exist)
final_order = (
    desired_order_start
    + incoming_cols
    + est_oh_cols
    + est_oo_cols
    + schedule_cols
    + amt_cols
    + ending_cols
)

final_order = [col for col in final_order if col in df_sum.columns]

# Apply reordering
df_sum = df_sum[final_order]


In [15]:
#TREND COEFFICIENT
import numpy as np

# --- STEP 1: Identify only D-1 to D-12 ---
d_cols = [
    col for col in PartsMovement_df.columns
    if isinstance(col, str) and col.startswith("D-") and col[2:].isdigit() and 1 <= int(col[2:]) <= 12
]

# Sort numerically: D-1, D-2, ... D-12
d_cols = sorted(d_cols, key=lambda x: int(x.split("-")[1]))

# Reverse to chronological order: D-12 → D-1
d_cols_reversed = list(reversed(d_cols))

# --- STEP 2: Compute Trend Coef (slope) ---
trend_list = []

for pn, group in PartsMovement_df.groupby("P/N"):
    # Sum all branch values for this PN
    values = group[d_cols_reversed].astype(float).sum(axis=0).values
    
    x = np.arange(len(values))  # 0..11 timeline
    
    slope = np.polyfit(x, values, 1)[0]  # linear regression slope
    
    trend_list.append([pn, slope])

trend_df = pd.DataFrame(trend_list, columns=["P/N", "Trend Coef"])

# --- STEP 3: Merge into df_sum ---
df_sum = df_sum.merge(trend_df, on="P/N", how="left")

# --- STEP 4: Insert Trend Coef after MM12 ---
insert_pos = df_sum.columns.get_loc("MM12") + 1
trend_series = df_sum.pop("Trend Coef")
df_sum.insert(insert_pos, "Trend Coef", trend_series)


In [16]:
# ============================================================
# MERGE ALERT COLUMNS
# ============================================================

if MULTIPLE_AGENCY:
    # Multiple agency: use NATIONAL + ALL AGC alert values.
    alert_filtered = Forecast_df[
        (Forecast_df["brc"] == "NATIONAL") &
        (Forecast_df["agc"] == "ALL AGC")
    ][["p/n", "forecast_alert_score", "forecast_alert_label"]]

    df_sum = df_sum.merge(
        alert_filtered,
        left_on="P/N",
        right_on="p/n",
        how="left"
    )

    df_sum = df_sum.drop(columns=["p/n"])

else:
    # Single agency: use the alert values directly by P/N.
    df_sum = df_sum.merge(
        Forecast_df[
            ["P/N", "forecast_alert_score", "forecast_alert_label"]
        ],
        on="P/N",
        how="left"
    )

# ============================================================
# ADD ADJUSTMENT COLUMNS
# BEFORE forecast_alert_score
# ============================================================
insert_position = df_sum.columns.get_loc("forecast_alert_score")

new_columns = [
    "Chk. OO",
    "Adj OO M-1",
    "Adj OO M-2",
    "Adj OO M-3",
    "Adj Sch.Ord M-1",
    "Adj Sch.Ord M-2",
    "Adj Sch.Ord M-3"
]

for idx, col_name in enumerate(new_columns):
    df_sum.insert(insert_position + idx, col_name, 0)


In [17]:
print("=" * 60)
print("PROCESSING COMPLETE")
print(f"Mode: {'MULTIPLE AGENCY' if MULTIPLE_AGENCY else 'SINGLE AGENCY'}")
print(f"Rows: {len(df_sum):,}")
print(f"RC Multipliers used: {RC_MULTIPLIER}")
print("=" * 60)
print(df_sum.columns.tolist())


Index(['P/N', 'Total Calls', 'RC', 'DN Price', 'FD_final', 'Max', 'OH', 'OO',
       'Incoming M-1', 'Incoming M-2', 'Incoming M-3', 'Incoming M-4',
       'Incoming M-5', 'Incoming M-6', 'Incoming M-7', 'Est. OH M-1',
       'Est. OH M-2', 'Est. OH M-3', 'Est. OH M-4', 'Est. OH M-5',
       'Est. OH M-6', 'Est. OH M-7', 'Est. OH M-8', 'Est. OH M-9',
       'Est. OO M-1', 'Est. OO M-2', 'Est. OO M-3', 'Est. OO M-4',
       'Est. OO M-5', 'Est. OO M-6', 'Est. OO M-7', 'Est. OO M-8',
       'Sched. Order M-1', 'Sched. Order M-2', 'Sched. Order M-3',
       'Sched. Order M-4', 'Sched. Order M-5', 'Sched. Order M-6',
       'Amt. Est. OO M-1', 'Amt. Est. OO M-2', 'Amt. Est. OO M-3',
       'Amt. Est. OO M-4', 'Amt. Est. OO M-5', 'Amt. Est. OO M-6',
       'Amt. Est. OO M-7', 'Amt. Est. OO M-8', 'Amt. Sched. Order M-1',
       'Amt. Sched. Order M-2', 'Amt. Sched. Order M-3',
       'Amt. Sched. Order M-4', 'Amt. Sched. Order M-5',
       'Amt. Sched. Order M-6', 'Desc', 'MM06', 'MM12', 'Tr

## Build the final 5-sheet workbook

The next cell creates the final Excel workbook and places all requested components into the five sheets above.

In [ ]:
# ============================================================
# BUILD FINAL 5-SHEET WORKBOOK
# ============================================================
# Sheet order:
# 1 Dashboard
# 2 Part Lookup
# 3 National Dashboard
# 4 FD Processed
# 5 pmovdcE
#
# Sheet 5 contains both the pmovdcE movement table (left) and the
# Dashboard Preliminary "Data" table (right).
# OPM / SL / PF are written as hardcoded values.
# LT / OCLT are live Excel formulas driven by editable parameter cells
# on the right side of Sheet 5. ExDlt / OC / Min / ROP / Max / Condition
# then recalculate live when LT/OCLT are changed.
# ============================================================

import os
import math
from io import BytesIO
import numpy as np
import pandas as pd
import openpyxl
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.chart import LineChart, Reference
from openpyxl.chart.label import DataLabelList
from openpyxl.chart.marker import Marker
from openpyxl.worksheet.formula import ArrayFormula

# ---------------------------
# Load forecast sheets needed by Dashboard/Data
# ---------------------------
forecast_branch = None
forecast_national = None
try:
    forecast_branch = pd.read_excel(FORECAST_FILE, sheet_name="Branch")
    forecast_branch.columns = [str(c).strip() for c in forecast_branch.columns]
except Exception as e:
    print(f"Branch forecast sheet not available: {e}")

try:
    forecast_national = pd.read_excel(FORECAST_FILE, sheet_name="National")
    forecast_national.columns = [str(c).strip() for c in forecast_national.columns]
except Exception as e:
    # Forecast_df from the PM process can still be used as a fallback.
    forecast_national = Forecast_df.copy()
    print(f"National forecast sheet fallback: {e}")

# ---------------------------
# Helper: normalize forecast columns
# ---------------------------
def normalize_forecast(df):
    if df is None:
        return None
    x = df.copy()
    rename = {}
    for c in x.columns:
        lc = str(c).strip().lower()
        if lc == "p/n": rename[c] = "PN"
        elif lc == "brc": rename[c] = "Brc"
        elif lc == "agc": rename[c] = "Agc"
    x = x.rename(columns=rename)
    if "PN" in x.columns:
        x["PN"] = x["PN"].astype(str).str.strip().str.upper()
    if "Brc" in x.columns:
        x["Brc"] = x["Brc"].astype(str).str.strip()
    if "Agc" in x.columns:
        x["Agc"] = x["Agc"].astype(str).str.strip()
    return x

fb = normalize_forecast(forecast_branch)
fn = normalize_forecast(forecast_national)

# ---------------------------
# Build Dashboard Preliminary-style Data table
# ---------------------------
pm = PartsMovement_df.copy()
pm.columns = [str(c).strip() for c in pm.columns]
def normalize_join_key(v):
    """Normalize Excel/Pandas join keys so 19, 19.0, and "19" match."""
    if pd.isna(v):
        return ""
    s = str(v).strip()
    if s.endswith(".0") and s[:-2].replace("-", "").isdigit():
        s = s[:-2]
    return s.upper()

pm["P/N"] = pm["P/N"].map(normalize_join_key)
if "Brc" in pm.columns:
    pm["Brc"] = pm["Brc"].map(normalize_join_key)
if "Agc" in pm.columns:
    pm["Agc"] = pm["Agc"].map(normalize_join_key)

call_cols = [f"C-{i}" for i in range(1, 13) if f"C-{i}" in pm.columns]
demand_cols = [f"D-{i}" for i in range(1, 13) if f"D-{i}" in pm.columns]
pm["Total_Calls"] = pm[call_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
pm["Total_Demands"] = pm[demand_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
pm["MM06"] = (pm[[f"C-{i}" for i in range(1,7) if f"C-{i}" in pm.columns]].apply(pd.to_numeric, errors="coerce").fillna(0) > 0).sum(axis=1)
pm["MM12"] = (pm[[f"C-{i}" for i in range(1,13) if f"C-{i}" in pm.columns]].apply(pd.to_numeric, errors="coerce").fillna(0) > 0).sum(axis=1)

# Collapse to one real branch + agency + PN row.
base_cols = ["Brc", "Agc", "P/N", "OH", "OO", "DN Price", "Desc", "Last Sales", "Last GRR",
             "Total_Calls", "Total_Demands", "MM06", "MM12"]
base_cols = [c for c in base_cols if c in pm.columns]
real = pm[base_cols].copy()

def first_nonblank(s):
    for v in s:
        if pd.notna(v) and str(v).strip() != "":
            return v
    return None

def longest_desc(s):
    vals = [str(v) for v in s if pd.notna(v) and str(v).strip()]
    return max(vals, key=len) if vals else ""

if len(real):
    grouped = real.groupby(["Brc", "Agc", "P/N"], as_index=False).agg({
        "OH":"sum", "OO":"sum", "DN Price":"first", "Desc":longest_desc,
        "Last Sales":"first", "Last GRR":"first", "Total_Calls":"sum", "Total_Demands":"sum",
        "MM06":"max", "MM12":"max"
    })
else:
    grouped = pd.DataFrame(columns=["Brc","Agc","P/N","OH","OO","DN Price","Desc","Last Sales","Last GRR","Total_Calls","Total_Demands","MM06","MM12"])

grouped = grouped.rename(columns={"P/N":"PN", "DN Price":"DN_Price", "Last Sales":"Last_Sales", "Last GRR":"Last_GRR"})

# Normalize all merge keys on the left side.
for _key in ["Brc", "Agc", "PN"]:
    if _key in grouped.columns:
        grouped[_key] = grouped[_key].map(normalize_join_key)

# Branch FD_final.
if fb is not None and {"Brc","Agc","PN","FD_final"}.issubset(fb.columns):
    branch_fd = fb[["Brc","Agc","PN","FD_final"]].copy()
    # Normalize merge keys to the same string representation as grouped.
    for _key in ["Brc", "Agc", "PN"]:
        branch_fd[_key] = branch_fd[_key].map(normalize_join_key)
    branch_fd = branch_fd.drop_duplicates(["Brc","Agc","PN"])
else:
    branch_fd = pd.DataFrame(columns=["Brc","Agc","PN","FD_final"])

grouped = grouped.merge(branch_fd, on=["Brc","Agc","PN"], how="left")

# Fallback to company-wide national forecast when branch forecast is unavailable.
if fn is not None and {"PN","FD_final"}.issubset(fn.columns):
    national_fd = fn[["PN","FD_final"]].copy()
    national_fd["PN"] = national_fd["PN"].map(normalize_join_key)
    national_fd = national_fd.drop_duplicates("PN").rename(columns={"FD_final":"FD_final_National"})
else:
    national_fd = pd.DataFrame(columns=["PN","FD_final_National"])

grouped = grouped.merge(national_fd, on="PN", how="left")
grouped["FD_final"] = grouped["FD_final"].fillna(grouped["FD_final_National"]).fillna(0)
grouped.drop(columns=["FD_final_National"], inplace=True)

# National rows per agency.
nat_base = grouped.groupby(["Agc","PN"], as_index=False).agg({
    "OH":"sum", "OO":"sum", "DN_Price":"first", "Desc":longest_desc,
    "Last_Sales":"max", "Last_GRR":"max", "Total_Calls":"sum", "Total_Demands":"sum",
    "MM06":"max", "MM12":"max", "FD_final":"first"
})
nat_base["Brc"] = "National"
nat_base = nat_base[["Brc","Agc","PN","OH","OO","DN_Price","Desc","Last_Sales","Last_GRR","Total_Calls","Total_Demands","MM06","MM12","FD_final"]]

# True ALL AGC national row.
grand = grouped.groupby("PN", as_index=False).agg({
    "OH":"sum", "OO":"sum", "DN_Price":"first", "Desc":longest_desc,
    "Last_Sales":"max", "Last_GRR":"max", "Total_Calls":"sum", "Total_Demands":"sum",
    "MM06":"max", "MM12":"max", "FD_final":"first"
})
grand["Brc"] = "National"
grand["Agc"] = "ALL AGC"
grand = grand[["Brc","Agc","PN","OH","OO","DN_Price","Desc","Last_Sales","Last_GRR","Total_Calls","Total_Demands","MM06","MM12","FD_final"]]

data = pd.concat([grouped, nat_base, grand], ignore_index=True)
data = data.drop_duplicates(["Brc","Agc","PN"]).reset_index(drop=True)

def calc_opm_sl_pf(calls, price):
    calls = float(calls or 0)
    price = float(price or 0)
    if calls <= 6: letter = "A"
    elif calls <= 11: letter = "B"
    elif calls <= 26: letter = "C"
    else: letter = "D"
    sl = 0
    for pmin, pmax, level in opm_quadrants[letter]["brackets"]:
        if price >= pmin and price < pmax:
            sl = level
            break
    pf = 0
    for k, v in zip(pf_lookup_values, pf_lookup_results):
        if sl >= k:
            pf = v
    return letter, sl, pf

ops = data.apply(lambda r: calc_opm_sl_pf(r["Total_Calls"], r["DN_Price"]), axis=1)
data["OPM"] = [x[0] for x in ops]
data["SL"] = [x[1] for x in ops]
data["PF"] = [x[2] for x in ops]

# ---------------------------
# Zone + condition helpers
# ---------------------------
branch_names = {
    21:"SEMARANG",22:"SURABAYA",23:"DENPASAR",24:"WAINGAPU",25:"MEDAN",26:"PADANG",27:"PEKANBARU",28:"JAMBI",30:"PALEMBANG",31:"LAMPUNG",
    32:"PONTIANAK",35:"MINING SUPPORT",36:"DEPO SAMARINDA",37:"BALIKPAPAN",38:"SAMARINDA",39:"BANJARMASIN",40:"BITUNG",41:"BARONG TONGKOK",
    43:"MOROWALI",45:"PALANGKA RAYA",46:"MUARA ENIM",47:"SINTANG",50:"CIKARANG",51:"CILEGON",52:"STAL KUDA",53:"BANDUNG",55:"PANGKAL PINANG",
    56:"BENETE",57:"BATU HIJAU",58:"TAPIN",61:"BENGKULU",62:"LAMPUNG TENGAH",63:"PAKUAN RATU",65:"SORONG",67:"TABANG",68:"PELALAWAN",69:"CILACAP",
    72:"TARAKAN",73:"KENDARI",74:"MAKASSAR",76:"SAMPIT",77:"MANADO",78:"ASEM REGES",79:"BENDILI",80:"BINTARO",81:"SANGATTA",82:"TANJUNG TABALONG",
    83:"BATU KAJANG",84:"BENGALON",87:"BERAU",88:"SATUI",91:"MELAK",92:"LATI",94:"TANJUNG PINANG",95:"KETAPANG",96:"KUPANG",97:"SUNGAI DANAU",98:"SEPARI"
}
zone_groups = {
    1:["MEDAN","PEKANBARU","JAMBI","PALEMBANG"],2:["PADANG","PEKANBARU"],3:["LAMPUNG","PALEMBANG"],4:["LAMPUNG","BENGKULU","PANGKAL PINANG","TANJUNG PINANG"],
    5:["BINTARO","ASEM REGES","CILEGON","CIKARANG","SORONG","SEMARANG","BANDUNG","SURABAYA"],6:["SURABAYA","DENPASAR","WAINGAPU","KUPANG","MAKASSAR","KENDARI"],
    7:["BITUNG","MANADO"],8:["BALIKPAPAN","SAMARINDA","MINING SUPPORT","DEPO SAMARINDA","TABANG","SANGATTA","BATU KAJANG","BENGALON","MELAK","LATI","SEPARI"],
    9:["BANJARMASIN","TAPIN","TANJUNG TABALONG","SATUI","PALANGKA RAYA","SAMPIT"],10:["PONTIANAK","SINTANG","KETAPANG"]
}
name_to_code = {v:k for k,v in branch_names.items()}
zone_map = {"National":0,20:0}
for z, names in zone_groups.items():
    for nm in names:
        if nm in name_to_code and name_to_code[nm] not in zone_map:
            zone_map[name_to_code[nm]] = z

def get_zone(brc):
    if isinstance(brc, str) and brc.isdigit():
        brc = int(brc)
    return zone_map.get(brc,99)

# ---------------------------
# Workbook + styles
# ---------------------------
wb = Workbook()
# remove default and create in required order
ws_dash = wb.active
ws_dash.title = "Dashboard"
ws_lookup = wb.create_sheet("Part Lookup")
ws_nat = wb.create_sheet("National Dashboard")
ws_fd = wb.create_sheet("FD Processed")
ws_mov = wb.create_sheet("pmovdcE")

FONT = "Arial"
NAVY = "1F2A3A"; LIGHT_GREY="F4F6F8"; MONTH_BLUE="D9E2EC"
WHITE_FONT = Font(color="FFFFFF", bold=True, size=13, name=FONT)
TITLE_FONT = Font(bold=True, size=12, name=FONT, color="1F4E78")
HEADER_FONT = Font(bold=True, color="FFFFFF", size=10, name=FONT)
BODY_FONT = Font(size=10, name=FONT)
BOLD_FONT = Font(bold=True, size=10, name=FONT)
LABEL_FONT = Font(bold=True, size=10, name=FONT, color="33414F")
INPUT_FONT = Font(bold=True, size=11, name=FONT)
TITLE_FILL = PatternFill("solid", fgColor=NAVY)
HEADER_FILL = PatternFill("solid", fgColor="1F4E78")
LIGHT_FILL = PatternFill("solid", fgColor=LIGHT_GREY)
MONTH_FILL = PatternFill("solid", fgColor=MONTH_BLUE)
INPUT_FILL = PatternFill("solid", fgColor="FFF6E5")
TOTAL_FILL = PatternFill("solid", fgColor="EDEFF2")
SECTION_FILL = PatternFill("solid", fgColor="33414F")
THIN = Side(style="thin", color="C9D2DA")
BORDER = Border(left=THIN,right=THIN,top=THIN,bottom=THIN)
GREEN_BORDER = Border(left=Side(style="medium",color="375623"),right=Side(style="medium",color="375623"),top=Side(style="medium",color="375623"),bottom=Side(style="medium",color="375623"))
CENTER = Alignment(horizontal="center", vertical="center")


def style_header(ws, row, c1, c2):
    for c in range(c1,c2+1):
        cell=ws.cell(row=row,column=c); cell.font=HEADER_FONT; cell.fill=HEADER_FILL; cell.alignment=CENTER; cell.border=BORDER

def setv(ws, ref, value, font=None, fill=None, align=None, numfmt=None, border=None):
    c=ws[ref]; c.value=value
    if font: c.font=font
    if fill: c.fill=fill
    if align: c.alignment=align
    if numfmt: c.number_format=numfmt
    if border: c.border=border
    return c

# ============================================================
# SHEET 4 — FD Processed
# ============================================================
df_export = df_sum.copy()
# Use an in-memory workbook buffer instead of /tmp so the notebook works on
# both Windows and Linux without assuming a /tmp directory exists.
fd_buffer = BytesIO()
df_export.to_excel(fd_buffer, index=False)
fd_buffer.seek(0)
raw_wb = openpyxl.load_workbook(fd_buffer, data_only=False)
raw_ws = raw_wb.active
for row in raw_ws.iter_rows():
    for cell in row:
        ws_fd[cell.coordinate] = cell.value
        cell2 = ws_fd[cell.coordinate]
        cell2.font = BODY_FONT
        cell2.border = BORDER
        if cell.row == 1:
            cell2.font = HEADER_FONT; cell2.fill = HEADER_FILL; cell2.alignment=CENTER
raw_wb.close()
fd_buffer.close()

headers_fd = {ws_fd.cell(1,c).value:c for c in range(1,ws_fd.max_column+1)}
for r in range(2, ws_fd.max_row+1):
    ohc=get_column_letter(headers_fd["OH"]); ooc=get_column_letter(headers_fd["OO"]); fdc=get_column_letter(headers_fd["FD_final"]); maxc=get_column_letter(headers_fd["Max"]); pricec=get_column_letter(headers_fd["DN Price"])
    if "Chk. OO" in headers_fd:
        chk=get_column_letter(headers_fd["Chk. OO"])
        ws_fd[f"{chk}{r}"] = f'=IF({ooc}{r}>=5*{maxc}{r},"Y","N")'
    for i in range(1,10):
        eoh=get_column_letter(headers_fd[f"Est. OH M-{i}"])
        if i<=7: inc=get_column_letter(headers_fd[f"Incoming M-{i}"])
        if i==1: ws_fd[f"{eoh}{r}"] = f"=({ohc}{r}+{ooc}{r}+{inc}{r})-{fdc}{r}"
        elif i<=7:
            peoh=get_column_letter(headers_fd[f"Est. OH M-{i-1}"]); peoo=get_column_letter(headers_fd[f"Est. OO M-{i-1}"])
            ws_fd[f"{eoh}{r}"] = f"=({peoh}{r}+{peoo}{r}+{inc}{r})-{fdc}{r}"
        else:
            peoh=get_column_letter(headers_fd[f"Est. OH M-{i-1}"]); peoo=get_column_letter(headers_fd[f"Est. OO M-{i-1}"])
            ws_fd[f"{eoh}{r}"] = f"=({peoh}{r}+{peoo}{r})-{fdc}{r}"
        if i<=8:
            eoo=get_column_letter(headers_fd[f"Est. OO M-{i}"])
            if i<=3 and "Chk. OO" in headers_fd:
                adj=get_column_letter(headers_fd[f"Adj OO M-{i}"]); chk=get_column_letter(headers_fd["Chk. OO"])
                ws_fd[f"{eoo}{r}"] = f'=IF({chk}{r}="Y",IF({eoh}{r}>{maxc}{r},0,{maxc}{r}-{eoh}{r})+{adj}{r},IF({eoh}{r}>{maxc}{r},0,{maxc}{r}-{eoh}{r}))'
            else:
                ws_fd[f"{eoo}{r}"] = f"=IF({eoh}{r}>{maxc}{r},0,{maxc}{r}-{eoh}{r})"
    for i in range(1,7):
        sc=get_column_letter(headers_fd[f"Sched. Order M-{i}"]); future=get_column_letter(headers_fd[f"Est. OH M-{i+3}"])
        if i<=3 and f"Adj Sch.Ord M-{i}" in headers_fd:
            adj=get_column_letter(headers_fd[f"Adj Sch.Ord M-{i}"])
            ws_fd[f"{sc}{r}"] = f"=IF({future}{r}>{maxc}{r},0,{maxc}{r}-{future}{r})+{adj}{r}"
        else:
            ws_fd[f"{sc}{r}"] = f"=IF({future}{r}>{maxc}{r},0,{maxc}{r}-{future}{r})"
    for i in range(1,9):
        eoo=get_column_letter(headers_fd[f"Est. OO M-{i}"]); amt=get_column_letter(headers_fd[f"Amt. Est. OO M-{i}"])
        ws_fd[f"{amt}{r}"] = f"={eoo}{r}*{pricec}{r}"
    for i in range(1,7):
        sc=get_column_letter(headers_fd[f"Sched. Order M-{i}"]); amt=get_column_letter(headers_fd[f"Amt. Sched. Order M-{i}"])
        ws_fd[f"{amt}{r}"] = f"={sc}{r}*{pricec}{r}"

for c in range(1,ws_fd.max_column+1): ws_fd.column_dimensions[get_column_letter(c)].width=max(11,min(24,len(str(ws_fd.cell(1,c).value or ""))+3))
ws_fd.freeze_panes="A2"

# ============================================================
# SHEET 5 — pmovdcE + Data
# ============================================================
ws_mov.sheet_view.showGridLines=False
movement_headers = ["Brc","P/N"] + [f"D-{i}" for i in range(1,17)] + [f"C-{i}" for i in range(1,13)]
for c,h in enumerate(movement_headers,1): setv(ws_mov,get_column_letter(c)+"1",h,LABEL_FONT,LIGHT_FILL,CENTER,border=BORDER)

# Real movement rows.
raw_move = pm.copy()
# Ensure required columns exist; fill missing with 0.
for h in movement_headers:
    if h not in raw_move.columns:
        raw_move[h]=0 if h.startswith(("D-","C-")) else ""
raw_move = raw_move[movement_headers].copy()
raw_move = raw_move[raw_move["P/N"].astype(str).str.strip().ne("")]

# National movement rows: one row per PN, aggregated across all real branches.
nat_move = raw_move.groupby("P/N",as_index=False)[[c for c in movement_headers if c.startswith(("D-","C-"))]].sum()
nat_move.insert(0,"Brc","National")
nat_move = nat_move[movement_headers]

move_all = pd.concat([raw_move,nat_move],ignore_index=True)
for r_idx,row in enumerate(move_all.itertuples(index=False,name=None),2):
    for c,v in enumerate(row,1): ws_mov.cell(r_idx,c,v).border=BORDER
move_last_row=len(move_all)+1
ws_mov.freeze_panes="A2"
ws_mov.column_dimensions["A"].width=11; ws_mov.column_dimensions["B"].width=18
for c in range(3,len(movement_headers)+1): ws_mov.column_dimensions[get_column_letter(c)].width=9

# Data table starts after a spacer column.
data_start_col=33  # AG
DATA_HEADERS=["Brc","Agc","PN","OH","OO","DN_Price","Desc","Last_Sales","Last_GRR","Total_Calls","Total_Demands","MM06","MM12","FD_final","LT","OCLT","OPM","SL","PF","ExDlt","OC","Min","ROP","Max","Zone","Condition","SortKey"]
for i,h in enumerate(DATA_HEADERS): setv(ws_mov,f"{get_column_letter(data_start_col+i)}1",h,HEADER_FONT,HEADER_FILL,CENTER,border=BORDER)
Dcol={h:data_start_col+i for i,h in enumerate(DATA_HEADERS)}

# Parameter area for live LT/OCLT.
param_col=data_start_col+len(DATA_HEADERS)+3
pc=get_column_letter(param_col); pc2=get_column_letter(param_col+1)
setv(ws_mov,f"{pc}1","LIVE LT / OCLT PARAMETERS",WHITE_FONT,TITLE_FILL,CENTER,border=BORDER)
setv(ws_mov,f"{pc}3","Branch",LABEL_FONT,LIGHT_FILL,CENTER,border=BORDER)
setv(ws_mov,f"{pc2}3","LT",LABEL_FONT,LIGHT_FILL,CENTER,border=BORDER)
branch_values=sorted([b for b in data["Brc"].dropna().unique() if b!="National"],key=lambda x:str(x))
branch_values += ["National"]
param_start=4
for i,b in enumerate(branch_values,param_start):
    setv(ws_mov,f"{pc}{i}",b,BODY_FONT,None,CENTER,border=BORDER)
    setv(ws_mov,f"{pc2}{i}",BRANCH_LT.get(b,""),INPUT_FONT,INPUT_FILL,CENTER,border=BORDER)
setv(ws_mov,f"{pc}2","OCLT",LABEL_FONT,LIGHT_FILL,CENTER,border=BORDER)
setv(ws_mov,f"{pc2}2",OCLT_VALUE,INPUT_FONT,INPUT_FILL,CENTER,border=BORDER)
lt_param_range=f"${pc}${param_start}:${pc2}${param_start+len(branch_values)-1}"
oclt_ref=f"${pc2}$2"

# Data values and calculations.
# The embedded Data table mirrors Dashboard Preliminary's Data-sheet structure.
# OPM / SL / PF are written as Excel formulas using the OPM / PF parameters
# configured in Cell 1. Therefore the calculation logic is preserved inside
# the pmovdcE sheet and is recalculated by Excel when the underlying row values
# change. Changing the parameter dictionaries in Cell 1 requires rerunning
# the notebook, just like the other generated hardcoded inputs.

def build_opm_formula(calls_ref):
    q = opm_quadrants
    return (
        f'=IF({calls_ref}<={q["A"]["calls_max"]},"A",'
        f'IF({calls_ref}<={q["B"]["calls_max"]},"B",'
        f'IF({calls_ref}<={q["C"]["calls_max"]},"C","D")))'
    )

def build_sl_for_quadrant(price_ref, brackets):
    expr = "0"
    # Build from the last bracket backwards so the first matching bracket wins.
    for pmin, pmax, level in reversed(brackets):
        expr = f'IF(AND({price_ref}>={pmin},{price_ref}<{pmax}),{level},{expr})'
    return expr

def build_sl_formula(opm_ref, price_ref):
    q = opm_quadrants
    sl_a = build_sl_for_quadrant(price_ref, q["A"]["brackets"])
    sl_b = build_sl_for_quadrant(price_ref, q["B"]["brackets"])
    sl_c = build_sl_for_quadrant(price_ref, q["C"]["brackets"])
    sl_d = build_sl_for_quadrant(price_ref, q["D"]["brackets"])
    return (
        f'=IFERROR(IF({opm_ref}="A",{sl_a},'
        f'IF({opm_ref}="B",{sl_b},'
        f'IF({opm_ref}="C",{sl_c},{sl_d}))),0)'
    )

pf_keys_excel = "{" + ",".join(str(int(v)) for v in pf_lookup_values) + "}"
pf_vals_excel = "{" + ",".join(str(v) for v in pf_lookup_results) + "}"

for i,row in data.reset_index(drop=True).iterrows():
    excel_row=i+2

    # Raw columns from the merged Data source.
    raw_fields = [
        "Brc","Agc","PN","OH","OO","DN_Price","Desc","Last_Sales","Last_GRR",
        "Total_Calls","Total_Demands","MM06","MM12","FD_final"
    ]
    for field in raw_fields:
        val=row.get(field,"")
        if pd.isna(val): val=""
        ws_mov.cell(excel_row,Dcol[field],val).font=BODY_FONT
        ws_mov.cell(excel_row,Dcol[field]).border=BORDER

    # Live LT/OCLT formulas.
    brc_ref=f"{get_column_letter(Dcol['Brc'])}{excel_row}"
    calls_ref=f"{get_column_letter(Dcol['Total_Calls'])}{excel_row}"
    price_ref=f"{get_column_letter(Dcol['DN_Price'])}{excel_row}"
    fd_ref=f"{get_column_letter(Dcol['FD_final'])}{excel_row}"

    lt_ref=f"{get_column_letter(Dcol['LT'])}{excel_row}"
    oclt_cell=f"{get_column_letter(Dcol['OCLT'])}{excel_row}"
    oclt_param_ref=oclt_ref

    ws_mov[lt_ref]=f'=IFERROR(VLOOKUP({brc_ref},{lt_param_range},2,FALSE),"")'
    ws_mov[oclt_cell]=f'={oclt_param_ref}'
    for ref in [lt_ref,oclt_cell]:
        ws_mov[ref].font=BODY_FONT
        ws_mov[ref].border=BORDER

    # OPM / SL / PF — same calculation chain as Dashboard Preliminary's Data sheet.
    opm_ref=f"{get_column_letter(Dcol['OPM'])}{excel_row}"
    sl_ref=f"{get_column_letter(Dcol['SL'])}{excel_row}"
    pf_ref=f"{get_column_letter(Dcol['PF'])}{excel_row}"

    ws_mov[opm_ref]=build_opm_formula(calls_ref)
    ws_mov[sl_ref]=build_sl_formula(opm_ref,price_ref)
    ws_mov[pf_ref]=f'=IFERROR(LOOKUP({sl_ref},{pf_keys_excel},{pf_vals_excel}),0)'

    for ref in [opm_ref,sl_ref,pf_ref]:
        ws_mov[ref].font=BODY_FONT
        ws_mov[ref].border=BORDER

    # ExDlt / OC / Min / ROP / Max — same downstream calculation as Data sheet.
    exd=f"{get_column_letter(Dcol['ExDlt'])}{excel_row}"
    oc=f"{get_column_letter(Dcol['OC'])}{excel_row}"
    min_ref=f"{get_column_letter(Dcol['Min'])}{excel_row}"
    rop_ref=f"{get_column_letter(Dcol['ROP'])}{excel_row}"
    max_ref=f"{get_column_letter(Dcol['Max'])}{excel_row}"

    ws_mov[exd]=f'=IFERROR(ROUND({fd_ref}*{lt_ref}/30,0),0)'
    ws_mov[oc]=f'=IFERROR(ROUND({oclt_cell}*{fd_ref}/14,0),0)'
    ws_mov[min_ref]=f'=ROUND({pf_ref}*SQRT({exd})+{oc},0)'
    ws_mov[rop_ref]=f'=ROUND({min_ref}+{exd},0)'
    ws_mov[max_ref]=f'=ROUND({min_ref}+{rop_ref},0)'

    # Zone is a hardcoded helper value, matching the Data sheet.
    zone_ref=f"{get_column_letter(Dcol['Zone'])}{excel_row}"
    ws_mov[zone_ref]=get_zone(row["Brc"])

    # Condition — same mutually-exclusive priority as the Data sheet.
    oh=f"{get_column_letter(Dcol['OH'])}{excel_row}"
    ls=f"{get_column_letter(Dcol['Last_Sales'])}{excel_row}"
    lg=f"{get_column_letter(Dcol['Last_GRR'])}{excel_row}"
    cond=f"{get_column_letter(Dcol['Condition'])}{excel_row}"
    sortkey=f"{get_column_letter(Dcol['SortKey'])}{excel_row}"

    deadstock_months = f'IF(OR({brc_ref}="National",{brc_ref}="20"),36,30)'
    latest_activity = f"MAX({ls},{lg})"
    deadstock_formula = (
        f'AND({oh}>0,{latest_activity}<>0,'
        f'{latest_activity}<EDATE(TODAY(),-{deadstock_months}))'
    )

    ws_mov[cond]=(
        f'=IFERROR(IF({deadstock_formula},"Deadstock",'
        f'IF(AND({oh}<>"",{oh}>{max_ref}),"Overstock",'
        f'IF({calls_ref}<4,"",'
        f'IF(AND({oh}<>"",{oh}<{exd}),"Critical",'
        f'IF(AND({oh}<>"",{oh}<{rop_ref}),"Reorder",""))))),"")'
    )

    # SortKey — same integer packing logic as the Data sheet.
    group_rank = f'IF(OR({cond}="Overstock",{cond}="Deadstock"),0,1)'
    ws_mov[sortkey]=(
        f"={zone_ref}*10000000000000+{group_rank}*1000000000000"
        f"+(999999-MIN({oh},999999))*1000000+ROW()"
    )

    for field in [
        "ExDlt","OC","Min","ROP","Max","Condition","SortKey","Zone"
    ]:
        cell=ws_mov.cell(excel_row,Dcol[field])
        cell.border=BORDER
        cell.font=BODY_FONT
# Format data columns and hide SortKey only.
for i,h in enumerate(DATA_HEADERS): ws_mov.column_dimensions[get_column_letter(data_start_col+i)].width=12 if h not in ("Desc","PN") else (28 if h=="Desc" else 18)
ws_mov.column_dimensions[get_column_letter(Dcol["SortKey"])].hidden=True

# Data ranges used by Dashboard / formulas.
DATA_FIRST=2; DATA_LAST=len(data)+1
R={h:f"'pmovdcE'!${get_column_letter(Dcol[h])}${DATA_FIRST}:${get_column_letter(Dcol[h])}${DATA_LAST}" for h in DATA_HEADERS}

# ============================================================
# SHEET 3 — National Dashboard
# ============================================================
ws_nat.sheet_view.showGridLines=False
setv(ws_nat,"A1","NATIONAL DASHBOARD",WHITE_FONT,TITLE_FILL,Alignment(horizontal="left",vertical="center"),border=BORDER)
ws_nat.merge_cells("A1:K2")
for row in ws_nat["A1:K2"]:
    for c in row: c.fill=TITLE_FILL
setv(ws_nat,"A4","FILTERS  (adjust Min / Max)",BOLD_FONT,SECTION_FILL,Alignment(horizontal="left"),border=BORDER); ws_nat.merge_cells("A4:F4")
for c,h in enumerate(["Field","Min","Max"],1): setv(ws_nat, ws_nat.cell(5,c).coordinate,h,LABEL_FONT,LIGHT_FILL,CENTER,border=BORDER)
filter_fields=[("Total Calls","Total_Calls","Total Calls"),("MM06","MM06","MM06"),("MM12","MM12","MM12")]
filter_refs={}
for i,(label,data_field,fd_field) in enumerate(filter_fields,6):
    vals=pd.to_numeric(data[data_field],errors="coerce").dropna() if data_field in data else pd.Series([0])
    setv(ws_nat,f"A{i}",label,LABEL_FONT,LIGHT_FILL,Alignment(horizontal="left"),border=BORDER)
    setv(ws_nat,f"B{i}",float(vals.min()) if len(vals) else 0,INPUT_FONT,INPUT_FILL,CENTER,"#,##0",BORDER)
    setv(ws_nat,f"C{i}",float(vals.max()) if len(vals) else 0,INPUT_FONT,INPUT_FILL,CENTER,"#,##0",BORDER)
    filter_refs[data_field]=(f"$B${i}",f"$C${i}")
criteria=[]
for _,data_field,fd_field in filter_fields: criteria.append((fd_field,*filter_refs[data_field]))
sec=10; setv(ws_nat,f"A{sec}","SUMMARY",HEADER_FONT,SECTION_FILL,Alignment(horizontal="left"),border=BORDER); ws_nat.merge_cells(f"A{sec}:F{sec}")
for c,h in enumerate(["PN Count","Total Amt. Est. OO","Total Amt. Sched. Order","Combined Total"],1): setv(ws_nat, ws_nat.cell(sec+1,c).coordinate,h,LABEL_FONT,LIGHT_FILL,CENTER,border=BORDER)
# Formula ranges against FD Processed.
fd_headers={ws_fd.cell(1,c).value:c for c in range(1,ws_fd.max_column+1)}

def fd_col(header): return get_column_letter(fd_headers[header])

# Monthly table.
table_sec=sec+3; setv(ws_nat,f"A{table_sec}","MONTHLY SUBTOTALS",HEADER_FONT,SECTION_FILL,Alignment(horizontal="left"),border=BORDER); ws_nat.merge_cells(f"A{table_sec}:K{table_sec}")
hr=table_sec+1; setv(ws_nat,f"A{hr}","Metric",LABEL_FONT,LIGHT_FILL,CENTER,border=BORDER)
months=[f"M-{i}" for i in range(1,10)]
for i,m in enumerate(months,2): setv(ws_nat, ws_nat.cell(hr,i).coordinate,m,LABEL_FONT,MONTH_FILL,CENTER,border=BORDER)
setv(ws_nat,f"K{hr}","Total",LABEL_FONT,LIGHT_FILL,CENTER,border=BORDER)
metric_rows={}; row=hr+1
for metric in ["Amt. Est. OO","Amt. Sched. Order"]:
    metric_rows[metric]=row; setv(ws_nat,f"A{row}",metric,LABEL_FONT,LIGHT_FILL,Alignment(horizontal="left"),border=BORDER)
    month_cells=[]
    for i,m in enumerate(months,2):
        colh=f"{metric} {m}"
        if colh in fd_headers:
            col=get_column_letter(i); sumrange=f"'FD Processed'!${fd_col(colh)}$2:${fd_col(colh)}${ws_fd.max_row}"
            parts=[]
            for field,lo,hi in criteria:
                rr=f"'FD Processed'!${fd_col(field)}$2:${fd_col(field)}${ws_fd.max_row}"
                parts += [f'{rr},">="&{lo}',f'{rr},"<="&{hi}']
            formula="=SUMIFS("+sumrange+","+",".join(parts)+")"
            setv(ws_nat,f"{col}{row}",formula,BODY_FONT,None,CENTER,"$#,##0",BORDER); month_cells.append(f"{col}{row}")
        else: setv(ws_nat,f"{get_column_letter(i)}{row}","",BODY_FONT,None,CENTER,border=BORDER)
    setv(ws_nat,f"K{row}","="+"+".join(month_cells),BOLD_FONT,TOTAL_FILL,CENTER,"$#,##0",BORDER); row+=1
combined=row; setv(ws_nat,f"A{combined}","Combined Total",BOLD_FONT,TOTAL_FILL,Alignment(horizontal="left"),border=BORDER)
for i in range(2,11): setv(ws_nat, ws_nat.cell(combined,i).coordinate,"="+"+".join(f"{get_column_letter(i)}{r}" for r in metric_rows.values()),BOLD_FONT,TOTAL_FILL,CENTER,"$#,##0",BORDER)
setv(ws_nat,f"K{combined}","="+"+".join(f"K{r}" for r in metric_rows.values()),BOLD_FONT,TOTAL_FILL,CENTER,"$#,##0",BORDER)
# summary formulas
pn_range=f"'FD Processed'!${fd_col('P/N')}$2:${fd_col('P/N')}${ws_fd.max_row}"
count_parts=[]
for field,lo,hi in criteria:
    rr=f"'FD Processed'!${fd_col(field)}$2:${fd_col(field)}${ws_fd.max_row}"; count_parts += [rr,f'">="&{lo}',rr,f'"<="&{hi}']
setv(ws_nat,f"A{sec+2}","=COUNTIFS("+",".join(count_parts)+")",BODY_FONT,None,CENTER,"#,##0",BORDER)
setv(ws_nat,f"B{sec+2}",f"=K{metric_rows['Amt. Est. OO']}",BODY_FONT,None,CENTER,"$#,##0",BORDER)
setv(ws_nat,f"C{sec+2}",f"=K{metric_rows['Amt. Sched. Order']}",BODY_FONT,None,CENTER,"$#,##0",BORDER)
setv(ws_nat,f"D{sec+2}",f"=K{combined}",BODY_FONT,None,CENTER,"$#,##0",BORDER)
ws_nat.column_dimensions["A"].width=24
for c in "BCDEFGHIJK": ws_nat.column_dimensions[c].width=13

# ============================================================
# SHEET 2 — Part Lookup
# ============================================================
ws_lookup.sheet_view.showGridLines=False
ws_lookup.merge_cells("A1:K2"); setv(ws_lookup,"A1","PART NUMBER LOOKUP",WHITE_FONT,TITLE_FILL,Alignment(horizontal="left",vertical="center"),border=BORDER)
for row in ws_lookup["A1:K2"]:
    for c in row: c.fill=TITLE_FILL
# PN and branch selectors.
setv(ws_lookup,"A4","Select P/N:",LABEL_FONT); setv(ws_lookup,"B4",str(df_sum.iloc[0]["P/N"]) if len(df_sum) else "",INPUT_FONT,INPUT_FILL,CENTER,border=BORDER)
setv(ws_lookup,"C4","Select Branch:",LABEL_FONT); setv(ws_lookup,"D4","National",INPUT_FONT,INPUT_FILL,CENTER,border=BORDER)
setv(ws_lookup,"E4","Description:",LABEL_FONT); ws_lookup.merge_cells("F4:K4")
# Hidden lookup lists on T:V.
unique_pn=sorted(pd.Series(data["PN"].dropna().astype(str)).unique().tolist())
unique_brc=["National"]+sorted([str(x) for x in data["Brc"].dropna().unique() if x!="National"],key=str)
for i,v in enumerate(unique_pn,2): ws_lookup[f"T{i}"]=v
for i,v in enumerate(unique_brc,2): ws_lookup[f"U{i}"]=v
ws_lookup.column_dimensions["T"].hidden=True; ws_lookup.column_dimensions["U"].hidden=True
pn_dv=DataValidation(type="list",formula1=f"='Part Lookup'!$T$2:$T${len(unique_pn)+1}",allow_blank=False); ws_lookup.add_data_validation(pn_dv); pn_dv.add(ws_lookup["B4"])
br_dv=DataValidation(type="list",formula1=f"='Part Lookup'!$U$2:$U${len(unique_brc)+1}",allow_blank=False); ws_lookup.add_data_validation(br_dv); br_dv.add(ws_lookup["D4"])
# Summary strip based on FD Processed.
setv(ws_lookup,"A6","PART SUMMARY",HEADER_FONT,SECTION_FILL,Alignment(horizontal="left"),border=BORDER); ws_lookup.merge_cells("A6:K6")
summary=[("Source","Agc"),("RC","RC"),("Total Calls","Total Calls"),("Unit Price","DN Price"),("FD Final","FD_final"),("Max","Max"),("On Hand","OH"),("On Order","OO"),("Trend Coef","Trend Coef"),("%Accum.","%Accum."),("Alert Score","forecast_alert_score"),("Alert","forecast_alert_label")]
for i,(label,field) in enumerate(summary,1): setv(ws_lookup, ws_lookup.cell(7,i).coordinate,label,LABEL_FONT,LIGHT_FILL,CENTER,border=BORDER)
# Source Agc from Sheet5 Data first matching PN.
data_pn_col=get_column_letter(Dcol["PN"]); data_agc_col=get_column_letter(Dcol["Agc"])
setv(ws_lookup,"A8",f'=IFERROR(INDEX(\'pmovdcE\'!${data_agc_col}$2:${data_agc_col}${DATA_LAST},MATCH($B$4,\'pmovdcE\'!${data_pn_col}$2:${data_pn_col}${DATA_LAST},0)),"Review")',BODY_FONT,None,CENTER,border=BORDER)
for i,(label,field) in enumerate(summary[1:],2):
    if field in fd_headers:
        col=get_column_letter(fd_headers[field]); formula=f'=IFERROR(INDEX(\'FD Processed\'!${col}:${col},MATCH($B$4,\'FD Processed\'!$A:$A,0)),"")'
    else: formula='=""'
    setv(ws_lookup, ws_lookup.cell(8,i).coordinate,formula,BODY_FONT,None,CENTER,border=BORDER)
# Monthly units and amounts.
unit_start=10; setv(ws_lookup,f"A{unit_start}","MONTHLY DETAIL QTY",HEADER_FONT,SECTION_FILL,Alignment(horizontal="left"),border=BORDER); ws_lookup.merge_cells(start_row=unit_start,start_column=1,end_row=unit_start,end_column=10)
for c,h in enumerate(["Metric"]+months,1): setv(ws_lookup, ws_lookup.cell(unit_start+1,c).coordinate,h,LABEL_FONT,MONTH_FILL if c>1 else LIGHT_FILL,CENTER,border=BORDER)
row=unit_start+2
for metric in ["Incoming","Est. OH","Est. OO","Sched. Order"]:
    setv(ws_lookup, ws_lookup.cell(row,1).coordinate,metric,LABEL_FONT,LIGHT_FILL,Alignment(horizontal="left"),border=BORDER)
    for j,m in enumerate(months,2):
        hdr=f"{metric} {m}"
        if hdr in fd_headers:
            c=get_column_letter(fd_headers[hdr]); formula=f'=IFERROR(INDEX(\'FD Processed\'!${c}:${c},MATCH($B$4,\'FD Processed\'!$A:$A,0)),"")'
        else: formula='=""'
        setv(ws_lookup, ws_lookup.cell(row,j).coordinate,formula,BODY_FONT,None,CENTER,"#,##0",BORDER)
    row+=1
units_last=row-1
amt_start=row+1; setv(ws_lookup,f"A{amt_start}","MONTHLY DETAIL",HEADER_FONT,SECTION_FILL,Alignment(horizontal="left"),border=BORDER); ws_lookup.merge_cells(start_row=amt_start,start_column=1,end_row=amt_start,end_column=10)
for c,h in enumerate(["Metric"]+months,1): setv(ws_lookup, ws_lookup.cell(amt_start+1,c).coordinate,h,LABEL_FONT,MONTH_FILL if c>1 else LIGHT_FILL,CENTER,border=BORDER)
row=amt_start+2
for metric in ["Amt. Est. OO","Amt. Sched. Order"]:
    setv(ws_lookup, ws_lookup.cell(row,1).coordinate,metric,LABEL_FONT,LIGHT_FILL,Alignment(horizontal="left"),border=BORDER)
    for j,m in enumerate(months,2):
        hdr=f"{metric} {m}"
        if hdr in fd_headers:
            c=get_column_letter(fd_headers[hdr]); formula=f'=IFERROR(INDEX(\'FD Processed\'!${c}:${c},MATCH($B$4,\'FD Processed\'!$A:$A,0)),"")'
        else: formula='=""'
        setv(ws_lookup, ws_lookup.cell(row,j).coordinate,formula,BODY_FONT,None,CENTER,"$#,##0.00",BORDER)
    row+=1
amount_last=row-1
# Demand/calls trend table based on pmovdcE raw area.
trend_start=amount_last+2; setv(ws_lookup,f"A{trend_start}","Call & Demand",HEADER_FONT,SECTION_FILL,Alignment(horizontal="left"),border=BORDER); ws_lookup.merge_cells(start_row=trend_start,start_column=1,end_row=trend_start,end_column=17)
trh=trend_start+1; periods=list(range(16,0,-1)); setv(ws_lookup,f"A{trh}","Months Ago",LABEL_FONT,LIGHT_FILL,CENTER,border=BORDER)
for j,p in enumerate(periods,2): setv(ws_lookup, ws_lookup.cell(trh,j).coordinate,p,LABEL_FONT,MONTH_FILL,CENTER,border=BORDER)
# helper formula sums selected PN + branch. National uses National rows only.
raw_brc_col="A"; raw_pn_col="B"
def mov_sum_formula(col_letter):
    return f'=IF($D$4="National",SUMIFS(\'pmovdcE\'!${col_letter}$2:${col_letter}${move_last_row},\'pmovdcE\'!$B$2:$B${move_last_row},$B$4,\'pmovdcE\'!$A$2:$A${move_last_row},"National"),SUMIFS(\'pmovdcE\'!${col_letter}$2:${col_letter}${move_last_row},\'pmovdcE\'!$B$2:$B${move_last_row},$B$4,\'pmovdcE\'!$A$2:$A${move_last_row},$D$4))'
rd=trh+1; setv(ws_lookup,f"A{rd}","Demand",LABEL_FONT,LIGHT_FILL,Alignment(horizontal="left"),border=BORDER)
rc=rd+1; setv(ws_lookup,f"A{rc}","Calls",LABEL_FONT,LIGHT_FILL,Alignment(horizontal="left"),border=BORDER)
for j,p in enumerate(periods,2):
    dcol=get_column_letter(2+p); setv(ws_lookup, ws_lookup.cell(rd,j).coordinate,mov_sum_formula(dcol),BODY_FONT,None,CENTER,"#,##0",BORDER)
    if p<=12:
        ccol=get_column_letter(18+p); setv(ws_lookup, ws_lookup.cell(rc,j).coordinate,mov_sum_formula(ccol),BODY_FONT,None,CENTER,"#,##0",BORDER)
# Trend chart.
chart=LineChart(); chart.title="Demand & Calls Trend"; chart.style=2; chart.width=24; chart.height=10; chart.y_axis.title="Demand / Calls"; chart.x_axis.title="Months Ago"
cats=Reference(ws_lookup,min_col=2,max_col=17,min_row=trh,max_row=trh)
ddata=Reference(ws_lookup,min_col=1,max_col=17,min_row=rd,max_row=rd); cdata=Reference(ws_lookup,min_col=1,max_col=17,min_row=rc,max_row=rc)
chart.add_data(ddata,titles_from_data=True,from_rows=True); chart.add_data(cdata,titles_from_data=True,from_rows=True); chart.set_categories(cats)
for s in chart.series: s.marker=Marker(symbol="circle",size=5)
chart.dataLabels=DataLabelList(); chart.dataLabels.showVal=True; chart.dataLabels.numFmt="#,##0"
ws_lookup.add_chart(chart,f"A{rc+3}")
ws_lookup.column_dimensions["A"].width=22
for c in range(2,18): ws_lookup.column_dimensions[get_column_letter(c)].width=13

# ============================================================
# SHEET 1 — Dashboard
# ============================================================
ws_dash.sheet_view.showGridLines=False
# selectors and hidden lists
setv(ws_dash,"A1","Agency",BOLD_FONT); setv(ws_dash,"B1",str(data["Agc"].iloc[0]) if len(data) else "",INPUT_FONT,INPUT_FILL,CENTER,border=GREEN_BORDER)
setv(ws_dash,"D1","Branch",BOLD_FONT); setv(ws_dash,"E1","National",INPUT_FONT,INPUT_FILL,CENTER,border=GREEN_BORDER)
setv(ws_dash,"G1","PN",BOLD_FONT); setv(ws_dash,"H1",str(data["PN"].iloc[0]) if len(data) else "",INPUT_FONT,INPUT_FILL,CENTER,border=GREEN_BORDER)
# branch 20
setv(ws_dash,"J1","OH (20)",BOLD_FONT); setv(ws_dash,"M1","OO (20)",BOLD_FONT)
# lists in T:V
agcs=sorted(pd.Series(data["Agc"].dropna().astype(str)).unique().tolist())
brcs=["National"]+sorted([str(x) for x in data["Brc"].dropna().unique() if x!="National"],key=str)
for i,v in enumerate(agcs,2): ws_dash[f"T{i}"]=v
for i,v in enumerate(brcs,2): ws_dash[f"U{i}"]=v
for i,v in enumerate(unique_pn,2): ws_dash[f"V{i}"]=v
for col in ["T","U","V"]: ws_dash.column_dimensions[col].hidden=True
for cell,formula in [("B1",f"='Dashboard'!$T$2:$T${len(agcs)+1}"),("E1",f"='Dashboard'!$U$2:$U${len(brcs)+1}"),("H1",f"='Dashboard'!$V$2:$V${len(unique_pn)+1}")]:
    dv=DataValidation(type="list",formula1=formula,allow_blank=False); ws_dash.add_data_validation(dv); dv.add(ws_dash[cell])
# description
setv(ws_dash,"G2","Desc",BOLD_FONT); ws_dash.merge_cells("H2:N2")
ws_dash["H2"]=f'=IFERROR(INDEX({R["Desc"]},MATCH($H$1,{R["PN"]},0)),"")'; ws_dash["H2"].fill=INPUT_FILL; ws_dash["H2"].border=GREEN_BORDER
# Output card
out_fields=[("OH","OH"),("OO","OO"),("DN Price","DN_Price"),("Total Calls","Total_Calls"),("Total Demands","Total_Demands"),("MM06","MM06"),("MM12","MM12"),("FD Final","FD_final"),("Min","Min"),("Max","Max"),("ExDlt","ExDlt"),("ROP","ROP"),("Last Sales","Last_Sales"),("Last GRR","Last_GRR")]
for i,(label,_) in enumerate(out_fields,1): setv(ws_dash, ws_dash.cell(3,i).coordinate,label,BOLD_FONT,None,CENTER,border=BORDER)
match=f'SUMPRODUCT(({R["Agc"]}=$B$1)*({R["Brc"]}=$E$1)*({R["PN"]}=$H$1)*ROW({R["PN"]}))'
for i,(label,field) in enumerate(out_fields,1):
    pos=ws_dash.cell(4,i); pos.value=f'=IFERROR(INDEX({R[field]},{match})-ROW({R[field].split(":")[0]})+1,"")' if False else f'=IFERROR(INDEX({R[field]},SUMPRODUCT(({R["Agc"]}=$B$1)*({R["Brc"]}=$E$1)*({R["PN"]}=$H$1)*(ROW({R["PN"]})-1))),"")'
    pos.fill=INPUT_FILL; pos.border=BORDER; pos.alignment=CENTER; pos.font=BODY_FONT
# Branch 20
br20=f'SUMPRODUCT(({R["Agc"]}=$B$1)*({R["Brc"]}="20")*({R["PN"]}=$H$1)*(ROW({R["PN"]})-1))'
ws_dash["K1"]=f'=IFERROR(INDEX({R["OH"]},{br20}),"")'; ws_dash["N1"]=f'=IFERROR(INDEX({R["OO"]},{br20}),"")'
# Condition/SOQ/Call Type
setv(ws_dash,"A5","Moving Agc",BOLD_FONT); setv(ws_dash,"B5","OH Agc",BOLD_FONT)
# Use TEXTJOIN array formulas.
formula_a6=f'=IF($H$1="","",_xlfn.TEXTJOIN(", ",TRUE,IF(({R["Brc"]}="National")*({R["Agc"]}<>"ALL AGC")*({R["PN"]}=$H$1)*({R["Total_Calls"]}>0),{R["Agc"]},"")))'
ws_dash["A6"]=ArrayFormula("A6", formula_a6)
formula_b6=f'=IF($H$1="","",_xlfn.TEXTJOIN(", ",TRUE,IF(({R["Brc"]}="National")*({R["Agc"]}<>"ALL AGC")*({R["PN"]}=$H$1)*({R["OH"]}>0),{R["Agc"]},"")))'
ws_dash["B6"]=ArrayFormula("B6", formula_b6)
setv(ws_dash,"A7","Condition",BOLD_FONT); ws_dash["B7"]=f'=IFERROR(INDEX({R["Condition"]},SUMPRODUCT(({R["Agc"]}=$B$1)*({R["Brc"]}=$E$1)*({R["PN"]}=$H$1)*(ROW({R["PN"]})-1))),"")'
setv(ws_dash,"D7","Call Type",BOLD_FONT); ws_dash["E7"]=f'=IF($D$4="","",IF($D$4>=4,"Auto","Low Auto"))'
setv(ws_dash,"G7","SOQ",BOLD_FONT); ws_dash["H7"]=f'=IF($B$7="Overstock",TEXT($A$4-$J$4,"#,##0")&" (Excess)",IF($B$7="Deadstock","-",IF($B$7="Critical",TEXT(MAX($L$4-$A$4-$B$4,0),"#,##0")&" (AF)",IF($B$7="Reorder",TEXT($J$4-$A$4-$B$4,"#,##0")&" (SF)",""))))'
# All branches table.
title=10; setv(ws_dash,f"A{title}","All Branches",TITLE_FONT)
headers_all=["Branch","Zone","PN","OH","OO","Total Calls","Total Demand","MM06","MM12","Min","Max","FD Final","ExDlt","ROP","Last Sales","Last GRR","Condition"]
for i,h in enumerate(headers_all,1): setv(ws_dash, ws_dash.cell(title+1,i).coordinate,h,HEADER_FONT,HEADER_FILL,CENTER,border=BORDER)
# Static dynamic-sort formulas based on SortKey. Keep table length bounded by number of branches.
max_rows=max(1,len(brcs)-1); start=title+2
for k in range(1,max_rows+1):
    rr=start+k-1
    qualify=f'({R["Agc"]}=$B$1)*({R["PN"]}=$H$1)*({R["Brc"]}<>$E$1)'
    # Helper hidden in R
    _r_formula=f'=IFERROR(MATCH(SMALL(IF({qualify},{R["SortKey"]}),{k}),{R["SortKey"]},0),"")'
    ws_dash[f"R{rr}"]=ArrayFormula(f"R{rr}", _r_formula)
    for j,field in enumerate(["Brc","Zone","PN","OH","OO","Total_Calls","Total_Demands","MM06","MM12","Min","Max","FD_final","ExDlt","ROP","Last_Sales","Last_GRR","Condition"],1):
        ws_dash.cell(rr,j).value=f'=IFERROR(INDEX({R[field]},$R{rr}),"")'; ws_dash.cell(rr,j).border=BORDER; ws_dash.cell(rr,j).font=BODY_FONT; ws_dash.cell(rr,j).alignment=CENTER
ws_dash.column_dimensions["R"].hidden=True
for c,w in {1:11,2:9,3:18,4:9,5:9,6:12,7:13,8:8,9:8,10:9,11:9,12:10,13:9,14:9,15:13,16:13,17:12}.items(): ws_dash.column_dimensions[get_column_letter(c)].width=w

# ============================================================
# Final workbook settings / formulas
# ============================================================
for ws in [ws_dash,ws_lookup,ws_nat,ws_fd,ws_mov]:
    ws.sheet_view.showGridLines=False
try:
    wb.calculation.fullCalcOnLoad = True
    wb.calculation.forceFullCalc = True
    wb.calculation.calcMode = "auto"
except Exception:
    pass

# Make the live parameter cells obvious.
ws_mov[f"{pc}1"].comment = openpyxl.comments.Comment("Edit LT values in the yellow cells and OCLT in the yellow cell. LT/OCLT in the Data table are live formulas.", "OpenAI")

# Save
wb.save(OUTPUT_FILE)
print("="*70)
print("FINAL WORKBOOK CREATED")
print(f"Mode: {'MULTIPLE AGENCY' if MULTIPLE_AGENCY else 'SINGLE AGENCY'}")
print(f"Output: {OUTPUT_FILE}")
print("Sheets:", wb.sheetnames)
print("Sheet 5 contains National pmovdcE rows + Dashboard Data.")
print("OPM / SL / PF are hardcoded. LT / OCLT and downstream inventory formulas are live.")
print("="*70)
